# 04. Hyperparameter Optimization & Final Model Evaluation — CampusWell RiskLab

## Section 1: Optimization Objectives & Search Methodology

Welcome to the **CampusWell RiskLab** Hyperparameter Optimization and Final Model Evaluation notebook.

### The Coarse-to-Fine Optimization Strategy
Rather than arbitrary trial-and-error, we execute a systematic two-stage search strategy on the training data ($N=1,600$):

1. **Stage 1 — Broad Exploration (`RandomizedSearchCV`):**
   Explores broad hyperparameter distributions across regularization strength $C$, tree depth, learning rate, and solver algorithms to identify promising convex regions in parameter space.
2. **Stage 2 — Focused Refinement (`GridSearchCV`):**
   Narrows search to a fine grid around the best parameter neighborhood identified in Stage 1 to pinpoint optimal convergence points.

### Objective Function & Constraint
All cross-validation maximizes **Weighted F1-score** under Stratified 5-Fold Cross-Validation:
$$\text{Maximize } F_1^{\text{weighted}} = \sum_{c \in \{\text{Low}, \text{Medium}, \text{High}\}} \frac{N_c}{N} F_{1, c}$$
The final champion model is also subjected to a clinical safety check: zero students with `High` risk should be misdiagnosed as `Low` risk.

## Section 2: Environment Setup & Data Loading

We import project configuration parameters, serialization utilities, metrics computation functions, and search modules.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

from src.config import (
    RANDOM_STATE,
    CV_FOLDS,
    TARGET_CLASSES,
    MODEL_PATH,
    MODEL_METADATA_PATH,
    FINAL_METRICS_PATH,
    OPTIMIZATION_RESULTS_PATH,
    REPORTS_FIGURES_DIR
)
from src.features.preprocessing import build_model_pipeline
from src.steps.preprocessing import load_splits
from src.steps.optimization import (
    run_randomized_search,
    run_grid_search,
    compare_with_baseline,
    select_production_candidate
)
from src.steps.evaluation import (
    train_final_pipeline,
    evaluate_on_test_set,
    analyze_errors,
    generate_evaluation_figures
)
from src.utils.serialization import save_pipeline, save_metadata
from src.utils.metrics import save_metrics, load_metrics

print("Optimization and evaluation environment successfully initialized.")

/home/badr-eddine-slaoui/Documents/CAMPUSWELL-RISΚLAB/models/production_pipeline.joblib
Pipeline(steps=[('feature_engineering', FeatureEngineer()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Year_of_Study',
                                                   'Sleep_Hours',
                                                   'Study_Hours_Per_Week',
                                                   'GPA', 'Social_Media_Hours',
                                                   'Screen_Time_Hours',
                                           

## Section 3: Training and Test Split Ingestion

We ingest the training partition ($N=1,600$) used for hyperparameter cross-validation, and the untouched test partition ($N=400$) reserved exclusively for final evaluation.

In [2]:
X_train, X_test, y_train, y_test = load_splits()
print(f"Training split samples: {len(X_train)} (Features: {X_train.shape[1]})")
print(f"Test split samples:     {len(X_test)} (Features: {X_test.shape[1]})")
print("\nTarget class counts in X_train:")
print(y_train.value_counts())
print("\nTarget class counts in X_test:")
print(y_test.value_counts())

Training split samples: 1600 (Features: 15)
Test split samples:     400 (Features: 15)

Target class counts in X_train:
Mental_Health_Risk
High      544
Medium    528
Low       528
Name: count, dtype: int64

Target class counts in X_test:
Mental_Health_Risk
High      136
Medium    132
Low       132
Name: count, dtype: int64


## Section 4: Hyperparameter Optimization Execution

We optimize the top two candidate architectures identified in the baseline benchmarking phase:
1. **Candidate 1 — Logistic Regression (Top Performer):**
   Hyperparameters tuned: $L_2$ inverse regularization parameter $C \in [0.01, 50.0]$, solver $\in [\text{'lbfgs'}, \text{'saga'}]$, and max iterations.
2. **Candidate 2 — HistGradientBoosting (Runner-Up Ensemble):**
   Hyperparameters tuned: learning rate $\eta \in [0.01, 0.2]$, max leaf nodes $\in [15, 63]$, min samples per leaf $\in [10, 30]$, max iterations, and $L_2$ regularization.